# [실습] 분류 모델 지표와 데이터 불균형

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🎯 [실습] 분류 지표와 불균형 — 어떤 실수를 감수할지 고른다

## — 0.89라는 숫자 뒤에, 놓친 사람이 몇 명인지 센다

지난 두 순서 동안 정확도만 봤습니다. 0.8451에서 0.8973까지 올렸고, 그 숫자가 흔들리지 않는다는 것도 확인했습니다. 그런데 한 번도 묻지 않은 질문이 있습니다 — **정확히 무엇을 맞히고 무엇을 틀렸나?**

> 🤖 **오늘의 AI 활용 규칙 — 검증 단계:**  
> 코드를 AI에게 물어도 됩니다. 단, **지표 이름을 정확히 지정**해서 묻고, 돌아온 숫자가 혼동행렬과 맞는지 직접 대조합니다. 지표를 잘못 고르는 것은 AI가 대신 책임져 주지 않습니다. (자세한 규칙은 개념 노트북 Part 0)

## 📋 오늘의 미션

마케팅팀이 구체적인 계획을 들고 왔습니다.

> 🧑‍💼 "구매로 이어질 것 같은 세션에 **무료배송 쿠폰**을 띄우려 합니다. 다음 달 예산으로 **500세션**까지 가능합니다. 어떤 세션을 골라야 할지, 그리고 그렇게 하면 **구매 건수를 몇 건이나 챙길 수 있는지** 알려주세요."

이 요청에는 두 가지가 들어 있습니다. **500이라는 용량 제약**과 **성과의 근거**입니다. 정확도로는 어느 쪽도 답할 수 없습니다. 오늘 그 답을 만듭니다.

| 문제 | 내용 | 확인하는 힘 |
| --- | --- | --- |
| 1 | 혼동행렬로 0.89를 쪼갠다 | 정확도의 착시 확인 |
| 2 | 지표 5종을 교차 검증으로 | 문제에 맞는 지표 고르기 |
| 3 | 임계값을 훑어 운영점 선택 | 용량 제약을 숫자로 번역 |
| 4 | `class_weight` 전후 비교 | 보정 채택 여부 결정 |
| 5 | 모델 카드 v4 완성 | 지표 선택 근거 기록 (**제출물**) |

> ⚠️ **미리 밝혀 둘 한계:**  
> 엄밀히 말하면 골라야 할 대상은 *"쿠폰이 있어야만 사는 세션"* 입니다. 그런데 우리 모델이 예측하는 것은 *"살 가능성이 높은 세션"* 이라 둘이 완전히 같지는 않습니다(개입의 효과를 예측하는 것은 더 어려운 문제입니다). 오늘은 후자로 진행하되, **이 차이를 모델 카드의 한계 항목에 적는 것**까지가 과제입니다.

# ⚙️ 데이터 준비

같은 데이터를 이어서 씁니다. 모델도 지난 순서에서 **채택한 것**을 동일하게 사용합니다 — 랜덤포레스트에 `min_samples_leaf=20`을 준 설정입니다(CV 정확도 0.8973, 학습-테스트 격차 0.0263).

오늘 바뀌는 것은 모델이 아니라 **모델을 읽는 방법**입니다.

| 데이터 | 내용 | 출처·라이선스 |
| --- | --- | --- |
| **Online Shoppers Purchasing Intention** | 온라인 스토어 방문 세션 12,330건 — 18개 열, 타깃 `Revenue`(구매 여부, 양성 15.5%) | [UCI 468](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset) · CC BY 4.0 |

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"구매 전환율(양성 비율): {y.mean():.4f}")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
쇼핑 세션 데이터: 12,330행 × 18열
구매 전환율(양성 비율): 0.1547

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. 혼동행렬로 0.89를 쪼갠다

정확도 0.8973은 "2,466건 중 약 2,213건을 맞혔다"는 뜻입니다. 그런데 **맞힌 것의 대부분이 '안 산다'** 였다면 이 정확도만으로 마케팅팀의 구매 세션 포착 목적을 충족했는지 판단할 수 없습니다. 네 칸으로 쪼개서 확인합니다.

```
[문제 1]
1) 지난 순서에서 채택한 모델을 학습합니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
2) 테스트셋에 대한 혼동행렬을 구해 TN · FP · FN · TP를 각각 출력합니다.
3) 정확도 · 정밀도 · 재현율 · F1을 계산해 함께 출력합니다.
4) "실제 구매 세션 중 몇 건을 놓쳤는가"를 건수와 비율로 설명합니다.
```

> 🤔 **예상하기**  
> 테스트셋 2,466건 중 실제 구매는 **382건**입니다. 정확도 0.89인 이 모델이 그 382건 중 몇 건이나 찾아낼 것 같습니까? 300건, 200건, 100건 가운데 예상값을 고릅니다.

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [2]:
# [C2] 문제 1. 혼동행렬로 0.89를 쪼갠다
# ⌨️ 문제 1 — 정확도 하나를 네 칸으로 쪼개기
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

# 1) 지난 순서와 동일한 분할
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) 지난 순서에서 채택한 모델 학습
model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=42
)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

# 3) 혼동행렬 → TN, FP, FN, TP
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print("[혼동행렬]")
print(f"TN (비구매 세션을 비구매으로 예측) : {tn}")
print(f"FP (비구매 세션을 구매로 오판)   : {fp}")
print(f"FN (구매 세션을 비구매으로 오판) : {fn}")
print(f"TP (구매 세션을 구매로 예측)   : {tp}")

# 4) 정확도 · 정밀도 · 재현율 · F1
acc = accuracy_score(y_te, pred)
prec = precision_score(y_te, pred)
rec = recall_score(y_te, pred)
f1 = f1_score(y_te, pred)

print("\n[성능 지표]")
print(f"정확도(Accuracy)  : {acc:.4f}")
print(f"정밀도(Precision) : {prec:.4f}")
print(f"재현율(Recall)    : {rec:.4f}")
print(f"F1 점수           : {f1:.4f}")

# 5) 놓친 구매 세션(FN) 건수와 비율
actual_positive = y_te.sum()
missed_ratio = fn / actual_positive

print("\n[놓친 구매 세션 분석]")
print(f"테스트셋의 실제 구매 세션 수     : {actual_positive}건")
print(f"모델이 놓친 구매 세션(FN) 수     : {fn}건")
print(f"놓친 비율 (FN / 실제 구매 세션)  : {missed_ratio:.4f} ({missed_ratio*100:.2f}%)")
print(f"\n→ 실제로 구매까지 이어진 세션 {actual_positive}건 중 {fn}건"
      f"({missed_ratio*100:.1f}%)을 모델이 '구매 안 함'으로 잘못 예측해 놓쳤습니다. "
      f"이는 재현율 {rec:.4f}과 같은 의미이며, 이 {fn}건에 해당하는 "
      f"잠재 구매 고객에게 개입할 기회를 놓친 셈입니다.")

[혼동행렬]
TN (비구매 세션을 비구매으로 예측) : 2002
FP (비구매 세션을 구매로 오판)   : 82
FN (구매 세션을 비구매으로 오판) : 186
TP (구매 세션을 구매로 예측)   : 196

[성능 지표]
정확도(Accuracy)  : 0.8913
정밀도(Precision) : 0.7050
재현율(Recall)    : 0.5131
F1 점수           : 0.5939

[놓친 구매 세션 분석]
테스트셋의 실제 구매 세션 수     : 382건
모델이 놓친 구매 세션(FN) 수     : 186건
놓친 비율 (FN / 실제 구매 세션)  : 0.4869 (48.69%)

→ 실제로 구매까지 이어진 세션 382건 중 186건(48.7%)을 모델이 '구매 안 함'으로 잘못 예측해 놓쳤습니다. 이는 재현율 0.5131과 같은 의미이며, 이 186건에 해당하는 잠재 구매 고객에게 개입할 기회를 놓친 셈입니다.


<details>
<summary>(클릭) 💡 힌트</summary>

- 분할은 지난 순서와 같아야 비교가 성립합니다 — `test_size=0.2, random_state=42, stratify=y`.
- `confusion_matrix(y_te, pred).ravel()`은 `(TN, FP, FN, TP)` 순서로 네 값을 한 번에 풀어줍니다.
- 놓친 구매 세션은 **FN**(실제 구매인데 아니라고 예측)입니다. 비율은 `FN / 실제 구매 세션 수`입니다.
- 실제 구매 세션 수는 `y_te.sum()`입니다.

</details>

> 🎯 **[C2] 확인하기**  
> **382건 중 196건입니다. 약 51.3%입니다.**
>
> 정확도 0.8913이 어떻게 만들어졌는지 보면 명확합니다. 맞힌 2,198건 중 **2,002건이 TN** — "안 살 사람을 안 산다고 맞힌 것"입니다. 전체의 84.5%가 원래 안 사는 사람이니, 다수 클래스를 잘 맞히는 것만으로도 정확도가 높게 나타날 수 있습니다.
>
> 마케팅팀의 구매 세션 포착 목적과 직접 연결되는 지표는 **재현율 0.5131** — 음성으로 분류한 실제 구매 세션 **186건**입니다. 지난 두 순서 동안 우리는 이 숫자를 **한 번도 본 적이 없습니다.**

# 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다

이제 지표를 늘립니다. 그리고 지난 순서에서 배운 대로, 한 번이 아니라 **5겹으로** 재서 평균 ± 표준편차로 적습니다.

불균형 데이터에서는 **평균 정밀도(Average Precision, AP)** (`average_precision`)를 함께 확인합니다. AP는 PR 곡선을 요약하지만 사다리꼴 적분으로 계산한 PR-AUC와 항상 같은 값은 아닙니다. 비교 대상이 되는 **기준선**도 함께 계산합니다 — 무작위 점수에서 기대되는 정밀도는 **양성 비율**입니다.

```
[문제 2]
1) cross_validate로 다섯 지표를 한 번에 구합니다.
   accuracy · precision · recall · f1 · average_precision(AP)
2) 각각 평균 ± 표준편차로 출력합니다.
3) AP를 양성 비율과 비교하고, 참고로 ROC-AUC도 구해 함께 확인합니다.
4) 다섯 지표 중 표준편차가 가장 큰 것이 무엇인지, 왜 그런지 설명합니다.
```

> 🤔 **예상하기**  
> 정확도의 표준편차는 지난 순서에서 **0.0035**였습니다. **재현율**의 표준편차는 그보다 클까요, 작을까요? (힌트: 각 검증 겹의 재현율은 약 382건의 양성 세션만 보고 계산됩니다)

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [3]:
# [C3] 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다
# ⌨️ 문제 2 — cross_validate로 다섯 지표를 동시에
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score

# 1) 5-겹 교차검증으로 다섯 지표를 한 번에 측정 (전체 X, y 사용)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model, X, y, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "average_precision"]
)

# 2) 지표별 평균 ± 표준편차 출력
metrics = ["accuracy", "precision", "recall", "f1", "average_precision"]
print("[5-겹 교차검증 결과 (평균 ± 표준편차)]")
means, stds = {}, {}
for m in metrics:
    key = f"test_{m}"
    means[m] = scores[key].mean()
    stds[m] = scores[key].std()
    print(f"{m:20s}: {means[m]:.4f} ± {stds[m]:.4f}")

# 3) AP를 양성 비율과 비교, ROC-AUC도 참고로 확인
baseline_ap = y.mean()
print(f"\n[AP vs 무작위 기준선]")
print(f"average_precision(AP) : {means['average_precision']:.4f}")
print(f"무작위 기준선(양성 비율) : {baseline_ap:.4f}")
print(f"→ AP가 기준선보다 {means['average_precision'] / baseline_ap:.1f}배 높음 "
      f"(모델이 무작위보다 훨씬 잘 순위를 매긴다는 뜻)")

proba = model.predict_proba(X_te)[:, 1]
roc_auc = roc_auc_score(y_te, proba)
print(f"\nROC-AUC (참고, 테스트셋 기준) : {roc_auc:.4f}")

# 4) 표준편차가 가장 큰 지표 확인 및 설명
worst_metric = max(stds, key=stds.get)
print(f"\n[표준편차가 가장 큰 지표]")
print(f"→ {worst_metric} (std={stds[worst_metric]:.4f})")
print(f"""
이유: 이 데이터셋은 양성(구매) 비율이 {y.mean():.2%}로 매우 낮은 불균형 데이터입니다.
{worst_metric}은(는) 분자·분모에 소수(양성 클래스)의 표본 수가 직접 관여하는 지표라,
겹(fold)마다 양성 샘플이 조금만 다르게 뽑혀도 값이 크게 흔들립니다.
반면 accuracy처럼 다수 클래스가 지배하는 지표는 폴드가 바뀌어도 상대적으로 안정적입니다.
""")

[5-겹 교차검증 결과 (평균 ± 표준편차)]
accuracy            : 0.8969 ± 0.0034
precision           : 0.7232 ± 0.0117
recall              : 0.5409 ± 0.0241
f1                  : 0.6186 ± 0.0175
average_precision   : 0.7248 ± 0.0191

[AP vs 무작위 기준선]
average_precision(AP) : 0.7248
무작위 기준선(양성 비율) : 0.1547
→ AP가 기준선보다 4.7배 높음 (모델이 무작위보다 훨씬 잘 순위를 매긴다는 뜻)

ROC-AUC (참고, 테스트셋 기준) : 0.9002

[표준편차가 가장 큰 지표]
→ recall (std=0.0241)

이유: 이 데이터셋은 양성(구매) 비율이 15.47%로 매우 낮은 불균형 데이터입니다.
recall은(는) 분자·분모에 소수(양성 클래스)의 표본 수가 직접 관여하는 지표라,
겹(fold)마다 양성 샘플이 조금만 다르게 뽑혀도 값이 크게 흔들립니다.
반면 accuracy처럼 다수 클래스가 지배하는 지표는 폴드가 바뀌어도 상대적으로 안정적입니다.



<details>
<summary>(클릭) 💡 힌트</summary>

- `cross_validate(model, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "average_precision"])`
- 결과는 `dict`입니다. 각 지표는 `결과["test_accuracy"]` 처럼 `test_` 접두사가 붙은 키에 들어 있습니다.
- `average_precision`은 AP를 계산합니다. PR-AUC와 목적은 유사하지만 적분 방식이 달라 값이 항상 같지는 않습니다.
- 무작위 점수에서 기대되는 정밀도는 `y.mean()`입니다 — 아무 정보 없이 무작위로 찍었을 때의 값입니다.
- ROC-AUC는 확률이 필요합니다: `roc_auc_score(y_te, proba)`.

</details>

> 🎯 **[C3] 확인하기**  
> **재현율의 표준편차가 0.0257로 가장 큽니다 — 정확도(0.0035)의 7배가 넘습니다.**
>
> 이유는 분모에 있습니다. 각 검증 겹에서 정확도는 약 2,466건 전체로 계산하지만, 재현율은 그중 약 382건인 양성 세션으로 계산합니다. 분모가 더 작으므로 겹별 구성 변화에 더 민감할 수 있습니다. 그래서 **소수 클래스 지표를 보고할 때는 표준편차를 함께** 적어야 합니다.
>
> 그리고 두 요약값을 확인합니다. **ROC-AUC는 0.9002이고 AP는 0.7248입니다.** 두 값은 축과 계산 방식이 달라 크기를 직접 비교하지 않습니다. ROC-AUC는 양성과 음성의 순위 구분을 요약하고, AP는 양성 예측의 정밀도와 재현율을 요약합니다. AP 0.7248은 양성 비율 0.1547보다 높지만, 채택 여부는 운영점의 정밀도·재현율과 비용을 함께 확인해 판단합니다.
>
> **어느 지표로 보고할 것인가.** 마케팅팀의 관심은 "실제 구매 세션을 얼마나 포착하는가"이므로 **재현율**과 **AP**가 주 지표이고, 쿠폰이 낭비되지 않는지를 보는 **정밀도**가 함께 갑니다. 정확도는 보조 지표로 제시하되 단독으로 결론을 내리지 않습니다.

# 문제 3. 임계값을 훑어 운영점을 정한다

여기까지의 숫자는 전부 **임계값 0.5**에서 나온 것입니다. 그런데 0.5는 그냥 기본값일 뿐, 쿠폰 용량이나 오류 비용을 반영해 선택한 값은 아닙니다. 마케팅팀의 제약은 **쿠폰 500장**입니다 — 그 제약을 임계값으로 번역합니다.

```
[문제 3]
1) 임계값을 0.10부터 0.70까지 훑으며 양성 예측 건수 · 정밀도 · 재현율 · F1을 표로 만듭니다.
2) F1이 가장 높은 임계값을 찾습니다.
3) "쿠폰 500장" 제약을 만족하는 임계값을 역산합니다.
   (확률 상위 500건만 고르려면 임계값이 얼마여야 하는가)
4) 그 운영점에 실제 구매 세션이 몇 건 포함되는지 계산해 마케팅팀에 보고합니다.
```

> 🤔 **예상하기**  
> 기본 임계값 0.5에서는 양성 예측이 **278건**뿐이었습니다(쿠폰 500장을 다 못 씁니다). 임계값을 낮춰 500장을 꽉 채우면, 포함되는 실제 구매 세션이 196건에서 몇 건까지 늘어날지 예상합니다.

> ⚠️ **주의하기 — 운영점 선택과 최종 평가 분리**  
> 이 실습은 한정된 데이터로 임계값 선택 과정을 연습하므로 홀드아웃 데이터에서 후보 운영점을 탐색합니다. 실제 업무에서는 검증 데이터 또는 교차 검증의 OOF 예측으로 임계값을 선택하고, 테스트 데이터는 최종 평가에 한 번만 사용합니다. 따라서 여기서 얻은 0.2695는 학습용 후보값이며 바로 배포할 최종값이 아닙니다.

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [4]:
# [C4] 문제 3. 임계값을 옮기면 무엇이 달라지는가
# ⌨️ 문제 3 — 임계값 스캔 + 쿠폰 500장 제약 역산
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# 테스트셋에 대한 예측 확률 (양성=구매 클래스)
proba = model.predict_proba(X_te)[:, 1]

# 1) 임계값 0.10 ~ 0.70을 훑으며 표 작성
thresholds = np.arange(0.10, 0.71, 0.05)
rows = []
for t in thresholds:
    pred_t = (proba >= t).astype(int)
    n_pos = pred_t.sum()
    prec = precision_score(y_te, pred_t, zero_division=0)
    rec = recall_score(y_te, pred_t, zero_division=0)
    f1 = f1_score(y_te, pred_t, zero_division=0)
    rows.append({"threshold": round(t, 2), "양성예측건수": n_pos,
                 "precision": prec, "recall": rec, "f1": f1})

table = pd.DataFrame(rows)
print("[임계값별 성능 표]")
print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# 2) F1이 가장 높은 임계값
best_row = table.loc[table["f1"].idxmax()]
print(f"\n[F1 최댓값 임계값]")
print(f"threshold={best_row['threshold']:.2f}, F1={best_row['f1']:.4f}, "
      f"precision={best_row['precision']:.4f}, recall={best_row['recall']:.4f}")

# 3) "쿠폰 500장" 제약을 만족하는 임계값 역산
N_COUPONS = 500
sorted_proba = np.sort(proba)[::-1]
t_coupon = sorted_proba[N_COUPONS - 1]  # 500번째로 높은 확률값
print(f"\n[쿠폰 500장 제약 임계값 역산]")
print(f"상위 500건 경계 임계값: {t_coupon:.4f}")

pred_coupon = (proba >= t_coupon).astype(int)
n_selected = pred_coupon.sum()
print(f"이 임계값으로 실제 선택되는 건수: {n_selected}건 "
      f"(동점자가 있으면 500건을 초과할 수 있음)")

# 4) 그 운영점에 포함된 실제 구매 세션 수 (TP)
tp_coupon = int(((pred_coupon == 1) & (y_te.values == 1)).sum())
precision_coupon = precision_score(y_te, pred_coupon, zero_division=0)
recall_coupon = recall_score(y_te, pred_coupon, zero_division=0)

print(f"\n[마케팅팀 보고용 요약]")
print(f"쿠폰 발송 대상(양성 예측) : {n_selected}건")
print(f"그중 실제 구매 세션(TP)  : {tp_coupon}건")
print(f"→ 쿠폰 500장을 확률 상위 {n_selected}건에 발송하면, "
      f"그중 {tp_coupon}건({precision_coupon:.1%})이 실제 구매 전환 세션이며 "
      f"전체 구매 세션의 {recall_coupon:.1%}를 커버합니다.")

[임계값별 성능 표]
 threshold  양성예측건수  precision  recall     f1
    0.1000     729     0.4595  0.8770 0.6031
    0.1500     595     0.5227  0.8141 0.6366
    0.2000     543     0.5506  0.7827 0.6465
    0.2500     506     0.5652  0.7487 0.6441
    0.3000     482     0.5747  0.7251 0.6412
    0.3500     449     0.5991  0.7042 0.6474
    0.4000     396     0.6338  0.6571 0.6452
    0.4500     337     0.6736  0.5942 0.6314
    0.5000     278     0.7050  0.5131 0.5939
    0.5500     216     0.7824  0.4424 0.5652
    0.6000     183     0.8251  0.3953 0.5345
    0.6500     135     0.8741  0.3089 0.4565
    0.7000     112     0.8750  0.2565 0.3968

[F1 최댓값 임계값]
threshold=0.35, F1=0.6474, precision=0.5991, recall=0.7042

[쿠폰 500장 제약 임계값 역산]
상위 500건 경계 임계값: 0.2683
이 임계값으로 실제 선택되는 건수: 500건 (동점자가 있으면 500건을 초과할 수 있음)

[마케팅팀 보고용 요약]
쿠폰 발송 대상(양성 예측) : 500건
그중 실제 구매 세션(TP)  : 284건
→ 쿠폰 500장을 확률 상위 500건에 발송하면, 그중 284건(56.8%)이 실제 구매 전환 세션이며 전체 구매 세션의 74.3%를 커버합니다.


<details>
<summary>(클릭) 💡 힌트</summary>

- 임계값을 적용한 예측은 `(proba >= t).astype(int)`입니다. `model.predict()`는 0.5로 고정이라 쓸 수 없습니다.
- 양성 예측 건수는 그 배열의 `.sum()`입니다.
- 정밀도 계산에서 양성 예측이 0건이면 오류가 납니다 — `precision_score(..., zero_division=0)`을 사용합니다.
- 상위 500건의 경계값은 확률을 **내림차순 정렬**해 500번째 값을 확인합니다: `np.sort(proba)[::-1][499]`. 경계에서 같은 점수가 여러 건이면 양성 예측이 500건을 넘을 수 있으므로 실제 운영에서는 동점 처리 규칙도 정합니다.
- 포함된 실제 구매 세션 수는 "양성 예측이면서 실제 양성"인 건수입니다 — 곧 **TP**입니다.

</details>

> 🎯 **[C4] 확인하기**  
> **포함되는 실제 구매 세션이 196건에서 284건으로 88건 늘어납니다.** 모델을 바꾸지도, 다시 학습시키지도 않았습니다. **임계값이라는 손잡이 하나**를 돌렸을 뿐입니다.
>
> 표를 세로로 확인합니다. 임계값이 내려갈수록 재현율은 오르고(0.2565 → 0.8770) 정밀도는 내려갑니다(0.8750 → 0.4595). 이 데이터에서는 두 지표 사이에 상충 관계가 나타납니다. 운영점은 모델 점수뿐 아니라 **용량과 비용 조건으로 정합니다.**
>
> F1이 가장 높은 지점은 0.20입니다. 하지만 우리가 고른 것은 **0.2695**입니다. 이는 학습용 홀드아웃에서 계산한 후보값이며, F1 최적점이 아니라 **예산이 감당하는 지점**입니다. 실무에서도 검증 데이터에서 이러한 운영 제약을 반영해 임계값을 선택합니다.

# 문제 4. `class_weight` 보정, 채택할 것인가

임계값 말고 다른 손잡이도 있습니다. **`class_weight="balanced"`** 는 학습 단계에서 소수 클래스에 가중치를 줘, 모델이 양성을 더 적극적으로 예측하게 만듭니다. 같은 목적에 도달하는 다른 길입니다.

두 방법은 작동 단계가 다르므로 **같은 검증 절차로 비교한 뒤 결정합니다.**

```
[문제 4]
1) class_weight="balanced"를 준 같은 모델로 교차 검증 지표 5종을 다시 구합니다.
2) 문제 2의 기본 설정과 나란히 표로 비교합니다.
3) 홀드아웃 혼동행렬도 함께 구해 TP·FN이 어떻게 달라졌는지 확인합니다.
4) 채택할 것인지 결정하고, 그 이유를 적습니다.
   (임계값 조정으로도 같은 효과를 낼 수 있다는 점을 함께 고려합니다)
```

> 🤔 **예상하기**  
> 개념 노트북의 데이터에서는 `class_weight`가 재현율을 크게 올리고 정밀도를 낮췄으며, F1은 소폭 올랐습니다. 이 데이터에서는 변화 폭이 어떻게 나타날지 예상합니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [5]:
# [C5] 문제 4. class_weight="balanced"가 정말 더 나은가
# ⌨️ 문제 4 — balanced 모델과 기본 모델 비교
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

# 문제 2에서 쓴 것과 같은 형식의 cv_report() 함수 (재사용)
def cv_report(model, X, y, cv, scoring):
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    report = {}
    for m in scoring:
        key = f"test_{m}"
        report[f"{m}_mean"] = scores[key].mean()
        report[f"{m}_std"] = scores[key].std()
    return report

scoring = ["accuracy", "precision", "recall", "f1", "average_precision"]

# 1) 기본 설정 vs class_weight="balanced" 설정, 나머지 하이퍼파라미터는 동일
base_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=42
)
balanced_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=42,
    class_weight="balanced"
)

base_report = cv_report(base_model, X, y, cv, scoring)
base_report["설정"] = "기본"
balanced_report = cv_report(balanced_model, X, y, cv, scoring)
balanced_report["설정"] = "balanced"

# 2) 두 설정을 나란히 표로 비교
compare_table = pd.DataFrame([base_report, balanced_report]).set_index("설정")
cols_order = [c for pair in zip([f"{m}_mean" for m in scoring],
                                  [f"{m}_std" for m in scoring]) for c in pair]
compare_table = compare_table[cols_order]
print("[기본 vs balanced 교차검증 비교]")
print(compare_table.to_string(float_format=lambda x: f"{x:.4f}"))

# 3) 홀드아웃 혼동행렬로 TP·FN 변화 확인
balanced_model.fit(X_tr, y_tr)
pred_balanced = balanced_model.predict(X_te)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_te, pred_balanced).ravel()

print("\n[홀드아웃 혼동행렬 비교 (문제 1의 기본 모델 vs balanced 모델)]")
print(f"기본     : TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"balanced : TN={tn_b}, FP={fp_b}, FN={fn_b}, TP={tp_b}")
print(f"→ TP 변화: {tp} → {tp_b} ({tp_b - tp:+d}) / FN 변화: {fn} → {fn_b} ({fn_b - fn:+d})")

# 4) 채택 여부 결정 및 이유
adopt = tp_b > tp and (fn - fn_b) > 0
print(f"\n[채택 여부]")
print(f"채택 여부: {'채택' if adopt else '보류'}")
print(f"""
이유:
- class_weight="balanced"는 학습 시 소수 클래스(구매)의 오분류에 더 큰 비용을 부여해
  recall/TP를 끌어올리는 대신 precision/FP가 늘어나는 경향이 있습니다.
- 하지만 문제 3에서 확인했듯, predict()의 기본 임계값(0.5)만 낮춰도
  동일한 방향(재현율↑, 정밀도↓)의 효과를 얻을 수 있습니다.
- class_weight를 바꾸는 것은 "학습 자체"를 바꾸는 결정이라 모델을 재학습해야 하고,
  임계값 조정은 "운영 시점"의 결정이라 배포된 모델은 그대로 두고 정책만 바꾸면 됩니다.
- 따라서 recall을 높이는 목적이라면, 굳이 class_weight를 바꾸기보다
  기본 모델 + 임계값 조정(문제 3의 방식)이 더 유연하고 되돌리기 쉬운 선택입니다.
- class_weight="balanced" 채택은, 교차검증 표에서 평균 성능(F1/AP 등)이
  기본 설정보다 유의미하게 더 좋고, 운영상 "고정된 단일 임계값"이 필요할 때만 고려합니다.
""")

[기본 vs balanced 교차검증 비교]
          accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  average_precision_mean  average_precision_std
설정                                                                                                                                                           
기본               0.8969        0.0034          0.7232         0.0117       0.5409      0.0241   0.6186  0.0175                  0.7248                 0.0191
balanced         0.8646        0.0051          0.5415         0.0109       0.8202      0.0234   0.6522  0.0125                  0.7176                 0.0158

[홀드아웃 혼동행렬 비교 (문제 1의 기본 모델 vs balanced 모델)]
기본     : TN=2002, FP=82, FN=186, TP=196
balanced : TN=1835, FP=249, FN=78, TP=304
→ TP 변화: 196 → 304 (+108) / FN 변화: 186 → 78 (-108)

[채택 여부]
채택 여부: 채택

이유:
- class_weight="balanced"는 학습 시 소수 클래스(구매)의 오분류에 더 큰 비용을 부여해
  recall/TP를 끌어올리는 대신 precision/FP가 늘어나는 경향이 있습니다.
- 하지만 문제 3에서 확인했듯, predict()의

<details>
<summary>(클릭) 💡 힌트</summary>

- 문제 2에서 만든 `cv_report()` 함수를 그대로 재사용하면 두 설정을 같은 형식으로 잴 수 있습니다.
- `RandomForestClassifier(..., class_weight="balanced")` 한 인자만 추가하면 됩니다. 나머지는 동일하게 둡니다 — 그래야 비교가 성립합니다.
- 두 결과 `dict`를 `pd.DataFrame([base, balanced])`로 묶으면 표가 됩니다.
- 혼동행렬은 문제 1과 같은 방식으로, `balanced` 모델을 `X_tr`에 학습시킨 뒤 구합니다.

</details>

> 🎯 **[C5] 확인하기**  
> 개념 노트북과 마찬가지로 `class_weight`가 재현율(0.5430 → 0.8097)뿐 아니라 **F1까지 올렸습니다**(0.6204 → 0.6571). 변화의 크기는 데이터와 모델에 따라 달라집니다.
>
> 그래서 "`class_weight`는 F1을 떨어뜨린다" 같은 규칙을 외우면 안 됩니다. **검증 데이터에서 지표와 운영 비용을 다시 측정하는 것**이 필요합니다.
>
> 그런데 F1이 올랐는데도 채택하지 않는 것이 합리적일 수 있습니다. **AP가 0.7248에서 0.7176로 소폭 낮아졌다는 사실**도 함께 확인합니다. 이 실험에서는 `class_weight`로 순위 요약 성능이 개선되지 않았습니다. 기본 모델의 임계값 조정으로 비슷한 재현율을 얻을 수 있지만, 두 방법이 항상 같은 예측을 만드는 것은 아닙니다. **지표 하나가 올랐다고 채택하는 것이 아니라, 무엇이 실제로 개선됐는지를 보고 결정합니다.**

# 문제 5. 모델 카드 v4 완성 — 오늘의 제출물

v4에는 **지표 선택 근거**와 **운영점**이 들어갑니다. 이제 모델 카드가 "성능 기록"을 넘어 **운영 문서**가 됩니다.

```
[문제 5]
1) 임계값 표를 파일로 남깁니다 — threshold_table_v4.csv
2) 아래 모델 카드 v4 템플릿의 빈칸을 채웁니다.
   '지표 선택 근거'와 '운영점 선택 근거'는 오늘의 핵심이니 숫자를 붙여 적습니다.
```

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [6]:
# [C6] 문제 5. 모델 카드 v4 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 임계값 표를 파일로 남기기

table.to_csv("threshold_table_v4.csv", index=False, encoding="utf-8-sig")
print("저장 완료: threshold_table_v4.csv")
print(table)

저장 완료: threshold_table_v4.csv
    threshold  양성예측건수  precision    recall        f1
0        0.10     729   0.459534  0.876963  0.603060
1        0.15     595   0.522689  0.814136  0.636643
2        0.20     543   0.550645  0.782723  0.646486
3        0.25     506   0.565217  0.748691  0.644144
4        0.30     482   0.574689  0.725131  0.641204
5        0.35     449   0.599109  0.704188  0.647413
6        0.40     396   0.633838  0.657068  0.645244
7        0.45     337   0.673591  0.594241  0.631433
8        0.50     278   0.705036  0.513089  0.593939
9        0.55     216   0.782407  0.442408  0.565217
10       0.60     183   0.825137  0.395288  0.534513
11       0.65     135   0.874074  0.308901  0.456480
12       0.70     112   0.875000  0.256545  0.396761


## 모델 카드 v4 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열, 양성 15.5%)
- 문제 유형: 분류 (불균형)
- 검증 방식: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 모델: RandomForest(n_estimators=300, min_samples_leaf=20)
- **주 지표와 선택 근거: 양성 비율이 15.5%에 불과한 불균형 데이터라, 정확도는 "전부 비구매로 예측"만 해도 84.5%가 나오는 함정 지표다. 그래서 F1(정밀도·재현율의 균형)과 AP(임계값에 의존하지 않는 순위 성능)를 주 지표로 본다.**
- **성능(CV, 평균 ± 표준편차)**
  - 정확도 0.8969 ± 0.0034 / 정밀도 0.7232 ± 0.0117 / 재현율 0.5409 ± 0.0241 / F1 0.6186 ± 0.0175 / AP 0.7248 ± 0.0191
  - **양성 비율 15.47%와 AP 0.7248 비교 → AP가 무작위 기준선보다 약 4.7배 높음**
- **혼동행렬(임계값 0.5): TN 2002 / FP 82 / FN 186 / TP 196**
  - **놓친 구매 세션 186건 (48.69%) ← 정확도 89.13%만 봤을 때는 보이지 않던 숫자. 실제 구매 고객의 절반 가까이를 놓치고 있었다는 뜻.**
- **운영점: 임계값 0.2683 (용량 제약 500건)**
  - **그 지점의 정밀도 0.568 / 재현율 0.743 → 실제 구매 세션 284건 포함**
  - **선택 근거: F1 최적점(threshold=0.35, 정밀도 0.599·재현율 0.704)은 통계적으로 가장 균형 잡힌 지점일 뿐, 실제 운영에서는 "쿠폰을 몇 장 찍을 수 있는가"라는 예산 제약이 우선한다. 500장이라는 리소스 한계가 먼저 정해져 있으므로, 그 한계 안에서 확률 상위 500명을 뽑는 것이 맞는 순서이지, F1이 최대인 지점을 찾은 뒤 예측 건수가 우연히 500건이 되길 바라는 것은 순서가 거꾸로다.**
- **class_weight 보정: 채택 미채택 — 근거: balanced 모델은 재현율을 0.5409→0.8202로 크게 올리지만 AP는 0.7248→0.7176으로 오히려 소폭 낮아진다(모델 자체의 순위 능력은 개선되지 않음). 반면 기본 모델에서 임계값만 0.15~0.20으로 낮추면 재현율 0.78~0.81·정밀도 0.52~0.55로 balanced 모델의 홀드아웃 결과(재현율 0.80·정밀도 0.55)와 사실상 동일한 지점에 도달한다. 즉 같은 효과를 재학습 없이 임계값 조정만으로 얻을 수 있고, 이 편이 배포된 모델은 그대로 둔 채 운영 정책만 되돌릴 수 있어 더 유연하다.**
- 한계 & 다음 단계: 모델이 예측하는 것은 "구매로 이어질 확률"이지 "쿠폰을 줬을 때 구매로 전환시키는 효과"가 아니다 — 이미 살 사람에게 쿠폰을 주는 것과 안 살 사람을 전환시키는 것은 다른 문제이므로, 인과 추론(uplift) 검증이 추가로 필요하다. 또한 VisitorType·Weekend·Month·TrafficType 등 범주형 8개 열을 아직 쓰지 않았으므로 추가 정보 손실 가능성이 있다. 마지막으로 지금의 운영점(임계값 0.2683)은 단일 홀드아웃 테스트셋 기준으로 정한 값이라 우연성이 섞여 있으므로, 실제 배포 전에는 교차검증의 OOF(out-of-fold) 예측이나 별도 검증셋에서 임계값을 다시 추정해야 한다.
- AI 사용 내역: 문제 1~5의 코드 작성(분할·학습·혼동행렬·교차검증·임계값 스캔·class_weight 비교·CSV 저장)을 Claude에게 요청했고, 실행과 실제 수치 산출은 직접 환경에서 진행했다. class_weight 채택 여부는 AI가 제시한 초안 결론과 실제 수치(AP 비교, 임계값 대안)를 대조해 직접 재검증한 뒤 결정했다.

**스스로 점검하는 기준**

| 축 | 기준 |
| --- | --- |
| 지표 정합 | 주 지표가 문제 상황(용량 제약·불균형)에 맞는가 |
| 비용 논리 | 용량 제약을 임계값으로 정확히 번역했는가 |
| 수치 근거 | 주장에 실제 계산값이 붙어 있는가 (감이 아니라 숫자) |
| 결정의 정직성 | 보정을 채택/기각한 이유가 지표 변화로 설명되는가 |

> 🚀 **직접 확장하기**  
> 쿠폰 원가가 2,000원이고 추가 구매 1건의 이익이 20,000원이라면, **이익을 최대로 만드는 임계값**은 얼마일까요? 임계값 표에 `이익 = TP × 20000 − 양성예측 × 2000` 열을 추가해 계산합니다. 용량 제약이 없을 때 비용이 어떻게 운영점을 정하는지 직접 확인할 수 있습니다.

오늘 여러분은 **0.89 뒤에 가려져 있던 실제 구매 세션 186건**을 찾아냈습니다. 그리고 손잡이 하나를 돌려 실제 구매 세션 88건을 더 포함하는 후보 운영안을 만들었습니다.

숫자를 올리는 일보다, 그 숫자가 **누구를 세고 있는지** 묻는 일이 먼저입니다.

오늘도 한 걸음, 수고하셨습니다! 🎉
어제의 나보다 데이터를 다루는 손이 한 뼘 더 능숙해졌습니다. 다음 시간에 또 만나요.

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>